# Imports

In [2]:
import pandas as pd
import pgeocode

# Data

In [ ]:
df = pd.read_parquet(
    './data/pp-complete.parquet',
    columns = ['postcode', 'town_city'],
    )
df.head()

In [ ]:
nomi = pgeocode.Nominatim("gb")

# запрос сразу по вектору — вернёт DataFrame с полями latitude/longitude и прочими
geo = nomi.query_postal_code(df["postcode"].tolist())

df["latitude"]  = geo["latitude"].values
df["longitude"] = geo["longitude"].values

df.drop(columns=['postcode'], inplace=True)
df.dropna(subset=["latitude", "longitude"], inplace=True)

df.head()

In [ ]:
cols = ["postcode", "latitude", "longitude"]
df.to_csv("output_with_coords.csv", columns=cols, index=False)

In [ ]:
from geopy.geocoders import Nominatim
import time

# Инициализируем геокодер
geolocator = Nominatim(user_agent="my_geocoder")

# Прогоним все уникальные города
city_coords = {}
nn = 0
for city in df['town_city'].unique():
    try:
        nn += 1
        loc = geolocator.geocode(f"{city}, UK")
        if loc:
            city_coords[city] = (loc.latitude, loc.longitude)
            print(f"Добавлен город {nn}/{len(df['town_city'].unique())}: {city}")
        else:
            city_coords[city] = (None, None)
            print(f"Ошибка при добавлении города {nn}/{len(df['town_city'].unique())}: {city}")
    except Exception:
        city_coords[city] = (None, None)
    #time.sleep(1)  # Нужная задержка между запросами (1 запрос в секунду)
    
# Распаковываем city_coords в колонки
df['city_lat'] = df['town_city'].map(lambda c: city_coords[c][0])
df['city_lon'] = df['town_city'].map(lambda c: city_coords[c][1])


In [ ]:
cols = ["town_city", "city_lat", "city_lon"]
df.to_csv("output_with_coords.csv", columns=cols, index=False)